# Drishti — End-to-End App Test (on Colab)

Runs the **real `app/` code** — router, engines, guardrail, translation, speech — against
real photos, on a Colab T4. This is the Phase-1 exit criterion, executed without installing
anything locally.

Everything in `app/` is unit-tested with fakes (124 tests). What has never happened is a
real model loading through it. That is what this notebook checks.

### Why the repo has to be fetched

The Colab VS Code extension runs *cells* on a Colab machine; it does not copy your project
there. `app/` therefore does not exist on the runtime until §1 fetches it — via `git clone`
(recommended) or a zip upload.

### OCR runs in a subprocess — read this before running

Medicine mode killed the Colab session twice: *"Your session crashed after using all
available RAM."* Not the PaddlePaddle/PyTorch collision of `DEC-006` — plain memory
exhaustion, from two causes stacking:

- **The kernel was full of things OCR does not need.** §2 used to `import transformers`,
  which brings in torch and (Colab preinstalls it) TensorFlow — 1–2 GB held for the whole
  Paddle phase. §2 now installs without importing.
- **The images were never downscaled.** `max_side=1600` only resizes when the longest side
  *exceeds* 1600, and the fixtures are exactly 1600×1204. So document unwarping — the
  heaviest stage — ran over the full 1.9 MP image. §4 now passes `--max-side 1280`.

§4 runs the OCR work in a **separate process**, which is independently what `DEC-006` called
the real fix. Two things follow:

- the OS reclaims every byte when it exits, so §6–§7 start from a clean baseline and **no
  manual runtime restart is needed** — run the notebook straight through
- if it still runs out of memory, the OOM killer takes the child (reported as exit `-9`)
  instead of your session. Lower `MAX_SIDE` in that cell and re-run **just that cell** —
  nothing above it is lost

**Runtime → Change runtime type → T4 GPU** before running.

In [1]:
import platform, os
print(platform.system(), platform.release())
print('hostname:', platform.node())
print('cwd:', os.getcwd())
!nvidia-smi --query-gpu=name --format=csv,noheader


Linux 6.6.122+
hostname: ef1892c9677e
cwd: /content
Tesla T4


## 1. Get the project onto the runtime

The Colab VS Code extension runs *cells* on Colab hardware but leaves your files on the
local disk, so `app/` does not exist on the runtime until we put it there.

> **`google.colab.files.upload()` does not work from VS Code.** It is a browser widget: the
> HTML renders, the JavaScript bridge that picks the file never loads, and the cell hangs
> until you interrupt it. Use one of the two paths below instead.

### Path A — git clone (recommended)

Push the project to GitHub once, then set `REPO_URL` below. Every later run is a single cell
that always pulls current code, and the repo ends up backed up and shareable with your guide.

```powershell
git remote add origin https://github.com/<you>/drishti.git
git push -u origin main
```

### Path B — run this notebook in the Colab browser

Open [colab.research.google.com](https://colab.research.google.com), upload this notebook,
and `files.upload()` behaves normally. Zero setup, but you lose the VS Code editor.

Sample photos are committed under `data/samples/`, so **no image upload is needed either
way** — that failure mode is gone entirely.

In [2]:
import os

# Set before any framework import -- see DEC-006.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import shutil, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/DevGurav/Drishti.git'
REFRESH = True         # re-fetch every run; set False only to keep a hand-edited runtime
PROJECT = Path('/content/drishti')
WORKDIR = Path('/content')

CLONE_HELP = """
If the repository is PRIVATE, clone with a token:
  GitHub -> Settings -> Developer settings -> Personal access tokens
  -> Fine-grained token, Repository access: this repo, Contents: Read
  REPO_URL = 'https://<TOKEN>@github.com/DevGurav/Drishti.git'

Otherwise: make the repo public, or use Path B (Colab browser + zip upload).
"""


def _find_project_root(start: Path):
    """Locate the folder holding app/router.py, however the archive nested it."""
    if (start / 'app' / 'router.py').exists():
        return start
    for marker in start.glob('*/app/router.py'):
        return marker.parents[1]
    return None


# Step out of PROJECT before deleting it. A re-run leaves the process cwd inside the
# project, and deleting the directory you are standing in makes every later subprocess
# fail with "unable to read current working directory" -- git exits 128.
os.chdir(WORKDIR)

# A stale copy is the failure that actually bites: the runtime keeps whatever was fetched
# first, so pushing new code changes nothing here and the error surfaces somewhere else.
if REFRESH:
    for stale in (PROJECT, WORKDIR / '_clone', WORKDIR / '_unpack'):
        shutil.rmtree(stale, ignore_errors=True)

if not (PROJECT / 'app' / 'router.py').exists():
    if REPO_URL:
        clone = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR / '_clone')],
            capture_output=True, text=True)
        if clone.returncode != 0:
            # Print git's own message. check=True hides stderr, and the cause is usually
            # only visible there (private repo, typo, auth).
            print(f'git clone failed (exit {clone.returncode}):')
            print(clone.stderr.strip())
            print(CLONE_HELP)
            raise SystemExit('clone failed -- see the message above')
        found = _find_project_root(WORKDIR / '_clone')
        if found is None:
            raise SystemExit('app/router.py not found in the cloned repo.')
        shutil.move(str(found), str(PROJECT))
    else:
        # Path B only -- this widget works in the Colab browser, never from VS Code.
        try:
            from google.colab import files
        except ImportError:
            raise SystemExit('Not running on Colab. Set REPO_URL above.')
        print('Upload drishti.zip  (Colab browser only; from VS Code set REPO_URL instead)')
        up = files.upload()
        with zipfile.ZipFile(next(iter(up))) as z:
            z.extractall(WORKDIR / '_unpack')
        found = _find_project_root(WORKDIR / '_unpack')
        if found is None:
            raise SystemExit('app/router.py not in the archive -- did you zip the drishti '
                             'folder itself?')
        shutil.move(str(found), str(PROJECT))

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

samples = sorted((PROJECT / 'data' / 'samples').glob('*.jpg'))
head = subprocess.run(['git', '-C', str(PROJECT), 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout.strip()

print('project root :', PROJECT)
print('fetched HEAD :', head or '(not a git checkout)')
print('modes        :', sorted(p.stem for p in (PROJECT / 'app' / 'modes').glob('[!_]*.py')))
print('sample images:', [p.name for p in samples] or 'NONE')

# Fail here, naming the real cause, rather than three cells later with a missing file.
if not samples:
    print('data/samples/ is empty, so the fetched code is out of date.')
    print('Most likely your local commits have not been pushed yet:')
    print('    git push')
    print('then re-run this cell (REFRESH=True forces a fresh clone).')
    raise SystemExit('stale code -- see the message above')

project root : /content/drishti
fetched HEAD : 5c651f9 docs: record the expiry-date bug and the paddle/torch split
modes        : ['ask', 'currency', 'medicine', 'read', 'scene']
sample images: ['strip_paracip.jpg', 'strip_partial.jpg']


In [3]:
# The suite needs no models, so a pass here proves the upload is complete and importable
# before we spend minutes downloading weights.
!python -m unittest discover -s tests -t . 2>&1 | tail -4

----------------------------------------------------------------------
Ran 124 tests in 0.126s

OK


## 2. Install engines

Weights are **not** downloaded here — every engine loads lazily on first use, so each mode
below pays only for what it needs.

> **Ignore this warning if you see it:** `gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0,
> but you have huggingface-hub 0.36.2 which is incompatible.` Colab's base image ships
> `gradio` pre-installed; installing `transformers`/`IndicTransToolkit` pins an older
> `huggingface-hub` than gradio declares it wants. Nothing in this project imports `gradio`,
> so the conflict is real but inert.

In [ ]:
# Install only -- deliberately no `import transformers` here.
#
# An earlier version verified the version by importing it in this cell. That put
# transformers, torch and (because Colab preinstalls it) TensorFlow into the kernel before
# §4 ran, costing 1-2 GB of RAM that the OCR phase has no use for. §4 then ran out of
# memory. The version check now lives in the torch-phase bootstrap, after OCR is done.
#
# transformers<5 is required by IndicTransToolkit, which imports PreTrainedTokenizerBase
# from transformers.tokenization_utils -- removed in v5. SmolVLM (§7) is not the reason for
# the pin: it already runs on 4.x, proven in notebook 00.
%pip install -q "transformers<5" paddlepaddle paddleocr IndicTransToolkit
print('installed -- transformers stays unimported until §6')

## 3. Choose test photos

Committed fixtures in `data/samples/` are used by default, so nothing needs uploading.

`strip_paracip.jpg` is the read that produced 55 OCR lines including the drug name,
`EXP.OCT.2026` and `Rs.10.30`. `strip_partial.jpg` is the same strip framed badly — only
3 lines — kept as the negative case.

To test **Devanagari Read mode**, drop a photo containing Marathi or Hindi text into
`data/samples/` and set `DEVANAGARI` below. That is still the one unverified claim in the
project: the code path is confirmed, the model has never seen actual Devanagari.

In [5]:
SAMPLES = PROJECT / 'data' / 'samples'
photos = sorted(SAMPLES.glob('*.jpg'))

for i, p in enumerate(photos):
    print(f'  [{i}] {p.name}  ({p.stat().st_size/1e3:.0f} KB)')

STRIP = SAMPLES / 'strip_paracip.jpg'      # the good read
DEVANAGARI = None                          # <-- set to a Marathi/Hindi photo to test §6

if not STRIP.exists():
    raise SystemExit(f'{STRIP} missing -- is data/samples/ present in the project?')

print('\nstrip     :', STRIP.name)
print('devanagari:', DEVANAGARI.name if DEVANAGARI else '(none set -- §6 will skip)')

  [0] strip_paracip.jpg  (320 KB)
  [1] strip_partial.jpg  (308 KB)

strip     : strip_paracip.jpg
devanagari: (none set -- §6 will skip)


## 4. Medicine mode — the guardrail, end to end

OCR reads the strip, the drug name is matched against the verified database, expiry and MRP
are parsed. If OCR cannot produce a verified name the mode **declines** rather than guessing
(`DEC-007`).

In [ ]:
%%writefile /content/ocr_phase.py
"""PaddlePaddle-only phase, run as its own process.

This ran in the notebook kernel until it exhausted Colab's RAM and killed the session --
"Your session crashed after using all available RAM", with everything above it lost. Two
things went wrong and both are fixed here.

1. It shared a process with the kernel, which by then held transformers, torch and
   TensorFlow (Colab preinstalls TF). None of that is needed to read a medicine strip.
   As a subprocess this imports only the OCR path, and when it exits the OS reclaims
   every byte before §6 loads the torch models. It also means an OOM kills *this*
   process, reported as exit -9, instead of silently taking down the notebook.

2. The default max_side of 1600 does not downscale the sample fixtures at all -- they are
   exactly 1600x1204, and the engine only resizes when the longest side *exceeds*
   max_side. So PaddleOCR ran document unwarping over the full 1.9 MP image. --max-side
   is exposed below so this is a knob you can turn rather than a constant to edit.

Peak RSS is printed so the next run produces a number instead of another guess.
"""
import argparse
import json
import os
import resource
import sys
import time
from pathlib import Path

PROJECT = Path('/content/drishti')
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

from app.drug_db import DrugDatabase
from app.engines.paddle_ocr import PaddleOCREngine
from app.modes.medicine import run as run_medicine
from app.modes.read import run as run_read


def peak_gb() -> float:
    """Peak RSS of this process. ru_maxrss is in kilobytes on Linux."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6


parser = argparse.ArgumentParser()
parser.add_argument('--strip', required=True, type=Path)
parser.add_argument('--devanagari', default=None, type=Path)
parser.add_argument('--out', required=True, type=Path)
parser.add_argument('--max-side', type=int, default=1280,
                    help='downscale before OCR. 1600 (the engine default) is a no-op on the '
                         '1600x1204 fixtures, which is what ran the runtime out of memory')
parser.add_argument('--fast', action='store_true',
                    help='disable document preprocessing. Cheapest memory win available, but '
                         'DEC-004 measured it destroying accuracy -- last resort, not a default')
args = parser.parse_args()

print(f'max_side={args.max_side}  fast={args.fast}')

t0 = time.time()
result = run_medicine(
    args.strip,
    PaddleOCREngine(lang='en', max_side=args.max_side, fast=args.fast),
    DrugDatabase.from_file(),
)
medicine_seconds = time.time() - t0

print(f'--- medicine mode  ({medicine_seconds:.1f}s, peak RSS {peak_gb():.2f} GB) ---')
print('verified :', result.ok)
print('drug     :', result.drug_name)
print('expiry   :', result.expiry_raw, '| expired:', result.expired)
print('MRP      :', result.mrp)
print('SPOKEN   :', result.message_en)

if not result.ok:
    print('\nDeclined. Either OCR missed the name, or it is absent from')
    print('data/drug_names_seed.txt -- a 30-entry placeholder, not a real drug database.')
    print('If --max-side was lowered, suspect the downscale before the database.')

# Read mode is PaddleOCR too, so it belongs in this process rather than a second one.
devanagari_text = devanagari_seconds = None
if args.devanagari:
    t0 = time.time()
    devanagari_text = run_read(
        args.devanagari, PaddleOCREngine(lang='mr', max_side=args.max_side, fast=args.fast))
    devanagari_seconds = round(time.time() - t0, 1)
    print(f'\n--- read mode, devanagari ({devanagari_seconds}s) ---')
    print(devanagari_text)
else:
    print('\nNo Devanagari photo passed -- skipping. Read mode in Marathi stays unverified.')

# ensure_ascii=False so Devanagari stays readable in the file rather than \uXXXX escapes.
args.out.write_text(json.dumps({
    'ok': result.ok,
    'drug_name': result.drug_name,
    'expiry_raw': result.expiry_raw,
    'expired': result.expired,
    'mrp': result.mrp,
    'message_en': result.message_en,
    'medicine_seconds': round(medicine_seconds, 1),
    'devanagari_text': devanagari_text,
    'devanagari_seconds': devanagari_seconds,
    'max_side': args.max_side,
    'fast': args.fast,
    'peak_rss_gb': round(peak_gb(), 2),
}, ensure_ascii=False), encoding='utf-8')
print(f'\npeak RSS {peak_gb():.2f} GB -- checkpoint written: {args.out}')

In [ ]:
import json
from pathlib import Path

CHECKPOINT = Path('/content/drishti_checkpoint.json')

MAX_SIDE = 1280   # 1600 is a no-op on the fixtures -- that is what ran out of RAM
FAST = False      # True disables doc preprocessing: big memory win, DEC-004 says big accuracy loss


def available_gb() -> float:
    for line in Path('/proc/meminfo').read_text().splitlines():
        if line.startswith('MemAvailable:'):
            return int(line.split()[1]) / 1e6
    return float('nan')


print(f'RAM available before OCR: {available_gb():.1f} GB')
print('(if this is already low, something above imported more than it needed --')
print(' §2 must not import transformers, and §4 must run before §6)\n')

# Run with `!` rather than subprocess.run so output streams into the cell as it happens --
# the first run downloads OCR models and is silent for a while, and a silent cell is
# indistinguishable from a hung one.
cmd = (f'python /content/ocr_phase.py --strip "{STRIP}" --out "{CHECKPOINT}" '
       f'--max-side {MAX_SIDE}' + (' --fast' if FAST else ''))
if DEVANAGARI:
    cmd += f' --devanagari "{DEVANAGARI}"'
print('$', cmd, '\n')
!{cmd}

if _exit_code != 0:
    # -9 is SIGKILL, which on Colab means the OOM killer. There is no traceback to read for
    # that -- the process was shot, it did not fail. Any other negative code is a different
    # signal; a positive code is a normal Python error with a traceback above.
    hint = ('the OOM killer -- lower MAX_SIDE (1024, then 896) and re-run this cell. The '
            'notebook itself survived, so nothing above needs re-running.'
            if _exit_code == -9 else 'see the output above.')
    raise SystemExit(f'OCR phase exited {_exit_code}: {hint}')

checkpoint = json.loads(CHECKPOINT.read_text(encoding='utf-8'))
print(f"\nsurvived. peak RSS {checkpoint['peak_rss_gb']} GB at max_side={checkpoint['max_side']}")
print('checkpoint:', checkpoint)

## 5. Read mode — Devanagari

Moved up next to medicine mode on purpose: both use **PaddleOCR only**, so they belong in
the same PaddlePaddle-only phase (see the restart notice below).

`lang='mr'` resolves to `devanagari_PP-OCRv5_mobile_rec`. The code path is confirmed; what
has never been tested is the model against actual Devanagari text.

In [ ]:
# Read mode already ran, inside the subprocess above -- it is PaddleOCR too, so it belongs
# in that process rather than a second one. Nothing to load here, just the result.
if checkpoint.get('devanagari_text') is None:
    print('No Devanagari photo set in §3 -- skipped. Read mode in Marathi stays unverified.')
else:
    print(f"--- read mode, devanagari ({checkpoint['devanagari_seconds']}s) ---")
    print(checkpoint['devanagari_text'])

## No restart needed — and why the earlier one didn't help

An earlier version ran OCR in the kernel and asked you to `Runtime → Restart session` here,
on the theory that PaddlePaddle and PyTorch were colliding (`DEC-006`). **That was the wrong
diagnosis.** Colab reported *"Your session crashed after using all available RAM"*: the
kernel ran out of memory during medicine mode, which a restart placed *after* the crash
could never prevent.

`DEC-006` still pointed at the right fix for the wrong reason — separate processes. §4 now
runs OCR as a subprocess, so its memory is reclaimed on exit and an OOM kills the child
rather than the session.

**Run the bootstrap cell below** — it defines `STRIP` and `checkpoint` for §6–§7 and is safe
to run whether or not the kernel was restarted.

In [ ]:
# Idempotent bootstrap for the torch phase. Safe to run whether or not the kernel was
# restarted -- it only restores names, and the VM's disk (packages, the clone, the
# checkpoint) survives a restart even though Python's in-memory state does not.
import json
import os
import sys
import time
from pathlib import Path

PROJECT = Path('/content/drishti')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# §3 may have been wiped by a restart, and §7 needs STRIP -- re-derive rather than re-running
# §3, which would be harmless now but is one more thing to remember.
SAMPLES = PROJECT / 'data' / 'samples'
STRIP = SAMPLES / 'strip_paracip.jpg'

CHECKPOINT = Path('/content/drishti_checkpoint.json')
if not CHECKPOINT.exists():
    raise SystemExit(
        f'{CHECKPOINT} not found. Did the OCR-phase cell (§4) finish? Scroll up and re-run it.'
    )

checkpoint = json.loads(CHECKPOINT.read_text(encoding='utf-8'))

# The version check §2 used to do. It lives here because importing transformers is what put
# torch and TensorFlow in the kernel during the OCR phase -- from this cell on, that is
# exactly what we want loaded. The sys.modules purge matters only if an earlier cell in this
# same kernel imported transformers before the pip downgrade took effect.
for _mod in [m for m in sys.modules if m == 'transformers' or m.startswith('transformers.')]:
    del sys.modules[_mod]
import transformers

print('transformers :', transformers.__version__, '(must be 4.x for IndicTransToolkit)')
print('project root :', PROJECT)
print('strip        :', STRIP.name, '(exists)' if STRIP.exists() else '(MISSING)')
print('checkpoint   :', checkpoint)

## 6. Marathi output and speech — the Phase-1 exit criterion

Reads `message_en` from the checkpoint the OCR subprocess wrote, rather than from a Python
variable — the OCR result was produced in a different process, so there is no variable to
read.

In [ ]:
from IPython.display import Audio, display

from app.engines.indictrans import IndicTrans2Translator
from app.engines.mms_tts import MMSTTSEngine
from app.speech import deliver

if not checkpoint.get('message_en'):
    raise SystemExit('checkpoint has no message_en -- did medicine mode (§4) succeed?')

translator = IndicTrans2Translator()
tts = MMSTTSEngine(out_dir=Path('/content/audio'))

for lang in ('mr', 'hi'):
    t0 = time.time()
    spoken = deliver(checkpoint['message_en'], lang=lang, translator=translator,
                     tts=tts, speak=True)
    print(f'--- {lang} ({time.time()-t0:.1f}s) ---')
    print(spoken.text_out)
    display(Audio(str(spoken.audio_path)))

## 7. Scene mode — the VLM

Same PyTorch-only session as §6 — no second restart needed, SmolVLM and IndicTransToolkit
coexist fine since both are PyTorch.

In [ ]:
from app.engines.smolvlm import SmolVLMEngine
from app.modes.ask import run as run_ask
from app.modes.scene import run as run_scene

vlm = SmolVLMEngine()

t0 = time.time()
print('--- scene mode ---')
print(run_scene(STRIP, vlm), f'({time.time()-t0:.1f}s)')

t0 = time.time()
print('\n--- ask mode ---')
print(run_ask(STRIP, vlm, 'what is written on this?'), f'({time.time()-t0:.1f}s)')

## 8. Findings — fill in, then update `docs/BUILD_PLAN.md`

| Check | Result | Latency |
|---|---|---|
| Medicine: drug name verified | | s |
| Medicine: expiry parsed (earliest of multiple dates) | | |
| Medicine: MRP parsed | | |
| Devanagari Read mode | | s |
| Runtime restart completed cleanly, checkpoint reloaded | | |
| Marathi translation readable | | s |
| Marathi speech intelligible | | s |
| Hindi speech intelligible | | s |
| Scene mode | | s |

**Phase 1 is complete when** the medicine row is verified and Marathi audio plays. Tick
those boxes in the build plan and record the latencies against the <8 s target (RISK-1) —
note that these numbers include the one-time PaddleOCR/PyTorch model download, so a repeat
run will be faster.

Ask a Marathi speaker whether the translation and the synthesized voice are actually
understandable — accuracy metrics do not capture intelligibility, and this is the first time
a human can judge the output.

**On the notebook design:** this run is why the notebook is now split into a PaddlePaddle
phase (§4–5) and a PyTorch phase (§6–7) with a mandatory restart between them, bridged by a
JSON checkpoint on disk. That split is not caution for its own sake — it is what fixed an
actual kernel crash reproduced during development (`DEC-006`, extended to cover
IndicTransToolkit as well as the VLM).